In [ ]:
import pandas as pd

In [8]:
events = pd.read_csv("../../data/raw/events.csv")
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [9]:
events.shape
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          str    
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 105.1 MB


In [10]:
events.isnull().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [12]:
events["event"].value_counts()


event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [13]:
events["visitorid"].nunique()


1407580

In [14]:
events["itemid"].nunique()

235061

In [15]:
events["transactionid"].notna().sum()

np.int64(22457)

In [16]:
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [25]:

events["datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms"
)

events["datetime"].min(), events["datetime"].max()

(Timestamp('2015-05-03 03:00:04.384000'),
 Timestamp('2015-09-18 02:59:47.788000'))

In [26]:
#customer events count
customer_event_counts = (
    events
    .groupby(["visitorid", "event"])
    .size()
    .unstack(fill_value=0)
)

customer_event_counts.head()

event,addtocart,transaction,view
visitorid,,,
0,0,0,3
1,0,0,1
2,0,0,8
3,0,0,1
4,0,0,1


In [27]:
#customers by event type
customer_event_counts.gt(0).sum()

event
addtocart        37722
transaction      11719
view           1404179
dtype: int64

In [28]:
#transactions
events.loc[
    events["event"] == "transaction",
    "transactionid"
].nunique()

17672

In [29]:
events.loc[
    events["event"] == "transaction",
    "visitorid"
].nunique()

11719

In [30]:
#cust summary
customer_summary = events.groupby("visitorid").agg(
    total_events=("event", "count"),
    unique_products=("itemid", "nunique"),
    first_activity=("datetime", "min"),
    last_activity=("datetime", "max")
)

customer_summary.describe()

,total_events,unique_products,first_activity,last_activity
count,1.407580e+06,1.407580e+06,1407580,1407580
mean,1.958042e+00,1.524019e+00,2015-07-09 10:16:42.513000,2015-07-11 19:17:24.858000
min,1.000000e+00,1.000000e+00,2015-05-03 03:00:04.384000,2015-05-03 03:00:11.289000
25%,1.000000e+00,1.000000e+00,2015-06-04 21:48:25.778000,2015-06-08 00:56:58.077000
50%,1.000000e+00,1.000000e+00,2015-07-10 10:54:21.271000,2015-07-13 01:08:21.987000
75%,2.000000e+00,1.000000e+00,2015-08-11 00:00:35.832000,2015-08-13 16:42:47.900000
max,7.757000e+03,3.814000e+03,2015-09-18 02:59:41.778000,2015-09-18 02:59:47.788000
std,1.258049e+01,7.143724e+00,NaN,NaN


In [32]:
events.head()

,timestamp,visitorid,event,itemid,transactionid,datetime
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106


In [33]:
customer_event_counts = (
    events
    .groupby(["visitorid", "event"])
    .size()
    .unstack(fill_value=0)
)

In [34]:
customer_features = customer_event_counts.reset_index()

customer_features = customer_features.rename(columns={
    "addtocart": "total_cart_adds",
    "transaction": "total_transactions",
    "view": "total_views"
})

In [35]:
unique_products = (
    events.groupby("visitorid")["itemid"]
    .nunique()
    .rename("unique_products")
)

customer_features = customer_features.merge(
    unique_products,
    on="visitorid",
    how="left"
)

In [36]:
customer_features.head()

,visitorid,total_cart_adds,total_transactions,total_views,unique_products
0,0,0,0,3,3
1,1,0,0,1,1
2,2,0,0,8,4
3,3,0,0,1,1
4,4,0,0,1,1


In [37]:
customer_features.groupby(
    customer_features["total_transactions"] > 0
)[[
    "total_views",
    "total_cart_adds",
    "total_transactions",
    "unique_products"
]].median()

,total_views,total_cart_adds,total_transactions,unique_products
total_transactions,,,,
False,1.0,0.0,0.0,1.0
True,4.0,1.0,1.0,2.0


In [38]:
customer_features.groupby(
    customer_features["total_transactions"] > 0
).size()

total_transactions
False    1395861
True       11719
dtype: int64

In [39]:
events.head()

,timestamp,visitorid,event,itemid,transactionid,datetime
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106


In [40]:
category_tree = pd.read_csv("../../data/raw/category_tree.csv")

category_tree.head()

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


In [41]:
category_tree.shape

(1669, 2)

In [42]:
category_tree.info()
category_tree.isnull().sum()
category_tree.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   categoryid  1669 non-null   int64  
 1   parentid    1644 non-null   float64
dtypes: float64(1), int64(1)
memory usage: 26.2 KB


,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0
5,231,NaN
6,542,378.0
7,1146,542.0
8,1140,542.0
9,1479,1537.0


In [43]:
item_properties_1 = pd.read_csv(
    "../../data/raw/item_properties_part1.csv"
)

item_properties_1.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


In [44]:
item_properties_1.info()

<class 'pandas.DataFrame'>
RangeIndex: 10999999 entries, 0 to 10999998
Data columns (total 4 columns):
 #   Column     Dtype
---  ------     -----
 0   timestamp  int64
 1   itemid     int64
 2   property   str  
 3   value      str  
dtypes: int64(2), str(2)
memory usage: 335.7 MB


In [45]:
item_properties_1["property"].value_counts().head(20)

property
888           1629817
790            970800
available      817387
categoryid     426305
6              343207
283            323681
776            311654
678            261829
364            256340
202            242984
839            226921
159            226502
917            226437
764            226242
112            226102
227            188209
698            157281
451            142388
663            131331
962            128976
Name: count, dtype: int64

In [46]:
item_properties_1[
    item_properties_1["property"] == "categoryid"
].head(10)

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
140,1432436400000,281245,categoryid,1277
151,1435460400000,35575,categoryid,1059
189,1437274800000,8313,categoryid,1147
197,1437879600000,55102,categoryid,47
213,1433041200000,397079,categoryid,619
237,1436670000000,265036,categoryid,1228
254,1437879600000,124459,categoryid,1277
310,1437879600000,350508,categoryid,546
325,1439089200000,221365,categoryid,1226


In [48]:
item_properties_2 = pd.read_csv(
    "../../data/raw/item_properties_part2.csv"
)

item_properties_2.head()

,timestamp,itemid,property,value
0,1433041200000,183478,561,769062
1,1439694000000,132256,976,n26.400 1135780
2,1435460400000,420307,921,1149317 1257525
3,1431831600000,403324,917,1204143
4,1435460400000,230701,521,769062


In [49]:
item_properties_2.shape

(9275903, 4)

In [50]:
item_properties_2[
    item_properties_2["property"] == "categoryid"
].shape

(361909, 4)

In [51]:
item_properties_2[
    item_properties_2["property"] == "categoryid"
].head()

,timestamp,itemid,property,value
15,1431226800000,8921,categoryid,1188
70,1433041200000,122405,categoryid,769
162,1439089200000,225336,categoryid,491
182,1435460400000,193256,categoryid,1261
257,1431226800000,301841,categoryid,1493


In [52]:
item_categories = pd.concat(
    [
        item_properties_1[
            item_properties_1["property"] == "categoryid"
        ],
        item_properties_2[
            item_properties_2["property"] == "categoryid"
        ]
    ],
    ignore_index=True
)

In [53]:
item_categories.groupby("itemid")["value"].nunique().describe()

count    417053.000000
mean          1.060853
std           0.259039
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: value, dtype: float64

In [54]:
(item_categories.groupby("itemid")["value"].nunique() > 1).sum()

np.int64(23352)

In [55]:
item_categories.groupby("itemid")["value"].nunique().sort_values(ascending=False).head(10)

itemid
42503     4
255468    4
391978    4
202195    4
231314    4
255846    4
383446    4
86370     4
146422    4
317831    4
Name: value, dtype: int64

In [56]:
item_categories["timestamp"] = pd.to_datetime(
    item_categories["timestamp"],
    unit="ms"
)

item_categories.head()

,timestamp,itemid,property,value
0,2015-06-28 03:00:00,460429,categoryid,1338
1,2015-05-24 03:00:00,281245,categoryid,1277
2,2015-06-28 03:00:00,35575,categoryid,1059
3,2015-07-19 03:00:00,8313,categoryid,1147
4,2015-07-26 03:00:00,55102,categoryid,47


In [57]:
item_categories.duplicated(
    subset=["itemid", "value"]
).sum()

np.int64(345782)

In [59]:
item_categories.columns

Index(['timestamp', 'itemid', 'property', 'value'], dtype='str')

In [60]:
item_categories = item_categories.rename(
    columns={"value": "categoryid"}
)

In [62]:
item_categories = item_categories[
    ["timestamp", "itemid", "categoryid"]
].copy()

In [63]:
item_categories.head()

,timestamp,itemid,categoryid
0,2015-06-28 03:00:00,460429,1338
1,2015-05-24 03:00:00,281245,1277
2,2015-06-28 03:00:00,35575,1059
3,2015-07-19 03:00:00,8313,1147
4,2015-07-26 03:00:00,55102,47


In [64]:
item_category_counts = (
    item_categories.groupby("itemid")["categoryid"]
    .nunique()
)

item_category_counts.value_counts().sort_index()

categoryid
1    393701
2     21373
3      1931
4        48
Name: count, dtype: int64

In [65]:
(item_category_counts > 1).sum()

np.int64(23352)

In [66]:
#remove duplicates
item_categories = (
    item_categories
    .drop_duplicates(subset=["itemid", "categoryid", "timestamp"])
    .sort_values(["itemid", "timestamp"])
    .reset_index(drop=True)
)

In [67]:
item_categories.shape

(788214, 3)

In [69]:
# ============================================================
# CUSTOMER EVENT → PRODUCT CATEGORY MAPPING
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Make sure timestamps are datetime
# ------------------------------------------------------------

# ============================================================
# TIME-AWARE CUSTOMER EVENT -> CATEGORY MAPPING
# ============================================================

# 1. Make sure timestamps are datetime
events["datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="ms"
)

item_categories["timestamp"] = pd.to_datetime(
    item_categories["timestamp"]
)

# 2. Select only required columns
events_for_merge = events[
    ["datetime", "visitorid", "event", "itemid", "transactionid"]
].copy()

category_history = item_categories[
    ["timestamp", "itemid", "categoryid"]
].copy()

# 3. Remove exact duplicate item/category/timestamp records
category_history = (
    category_history
    .drop_duplicates(
        subset=["itemid", "categoryid", "timestamp"]
    )
)

# 4. IMPORTANT:
# merge_asof requires the merge time columns to be globally sorted.
events_for_merge = (
    events_for_merge
    .sort_values(["datetime", "itemid"])
    .reset_index(drop=True)
)

category_history = (
    category_history
    .sort_values(["timestamp", "itemid"])
    .reset_index(drop=True)
)

# 5. Time-aware merge
# For each event, find the latest category assignment
# for that item occurring at or before the event time.

enriched_events = pd.merge_asof(
    events_for_merge,
    category_history,
    left_on="datetime",
    right_on="timestamp",
    by="itemid",
    direction="backward"
)

# 6. Remove the category-history timestamp
enriched_events = enriched_events.drop(
    columns=["timestamp"]
)

# 7. Restore chronological order
enriched_events = (
    enriched_events
    .sort_values(["visitorid", "datetime"])
    .reset_index(drop=True)
)

# 8. Diagnostics
print("Original events:", len(events))
print("Enriched events:", len(enriched_events))

missing_categories = enriched_events["categoryid"].isna().sum()
total_events = len(enriched_events)

print("\nEvents without category:", missing_categories)
print(
    "Events with category:",
    total_events - missing_categories
)

print(
    "Category coverage:",
    round(
        enriched_events["categoryid"].notna().mean() * 100,
        2
    ),
    "%"
)

print("\nSample enriched events:")
display(enriched_events.head(10))

Original events: 2756101
Enriched events: 2756101

Events without category: 656928
Events with category: 2099173
Category coverage: 76.16 %

Sample enriched events:


,datetime,visitorid,event,itemid,transactionid,categoryid
0,2015-09-11 20:49:49.439,0,view,285930,NaN,1188
1,2015-09-11 20:52:39.591,0,view,357564,NaN,256
2,2015-09-11 20:55:17.175,0,view,67045,NaN,333
3,2015-08-13 17:46:06.444,1,view,72028,NaN,1192
4,2015-08-07 17:51:44.567,2,view,325215,NaN,299
5,2015-08-07 17:53:33.790,2,view,325215,NaN,299
6,2015-08-07 17:56:52.664,2,view,259884,NaN,299
7,2015-08-07 18:01:08.920,2,view,216305,NaN,299
8,2015-08-07 18:08:25.669,2,view,342816,NaN,444
9,2015-08-07 18:17:24.375,2,view,342816,NaN,444


In [70]:
# ============================================================
# MULTI-CATEGORY CUSTOMER AFFINITY
# ============================================================

# 1. Keep events where category information is available
affinity_events = enriched_events.dropna(
    subset=["categoryid"]
).copy()

# 2. Make category IDs integer-like
affinity_events["categoryid"] = (
    affinity_events["categoryid"]
    .astype(int)
)

# 3. Assign behavioural weights
#    View       = 1
#    Add to cart = 3
#    Transaction = 5
event_weights = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0
}

affinity_events["interaction_weight"] = (
    affinity_events["event"]
    .map(event_weights)
    .fillna(0)
)

# 4. Calculate customer-category weighted interaction score
customer_category_affinity = (
    affinity_events
    .groupby(["visitorid", "categoryid"])
    ["interaction_weight"]
    .sum()
    .reset_index()
)

# 5. Calculate total affinity score for each customer
customer_category_affinity["customer_total_score"] = (
    customer_category_affinity
    .groupby("visitorid")["interaction_weight"]
    .transform("sum")
)

# 6. Normalize scores within each customer
#    This produces a 0-1 affinity distribution across categories.
customer_category_affinity["affinity_score"] = (
    customer_category_affinity["interaction_weight"]
    / customer_category_affinity["customer_total_score"]
)

# 7. Sort each customer's categories by affinity
customer_category_affinity = (
    customer_category_affinity
    .sort_values(
        ["visitorid", "affinity_score"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

# 8. Display results
print("Customer-category affinity rows:",
      len(customer_category_affinity))

print(
    "Unique customers:",
    customer_category_affinity["visitorid"].nunique()
)

print(
    "Unique categories:",
    customer_category_affinity["categoryid"].nunique()
)

print("\nSample affinity table:")
display(customer_category_affinity.head(20))

Customer-category affinity rows: 1245375
Unique customers: 1076473
Unique categories: 1139

Sample affinity table:


,visitorid,categoryid,interaction_weight,customer_total_score,affinity_score
0,0,256,1.0,3.0,0.333333
1,0,333,1.0,3.0,0.333333
2,0,1188,1.0,3.0,0.333333
3,1,1192,1.0,1.0,1.000000
4,2,299,6.0,8.0,0.750000
5,2,444,2.0,8.0,0.250000
6,3,1171,1.0,1.0,1.000000
7,5,646,1.0,1.0,1.000000
8,6,342,8.0,8.0,1.000000
9,7,642,1.0,2.0,0.500000


In [71]:
# ============================================================
# CUSTOMER CATEGORY AFFINITY - EVIDENCE + CONFIDENCE
# ============================================================

# Number of raw interactions behind each customer-category pair
customer_category_affinity["interaction_count"] = (
    affinity_events
    .groupby(["visitorid", "categoryid"])
    .size()
    .reindex(
        pd.MultiIndex.from_frame(
            customer_category_affinity[
                ["visitorid", "categoryid"]
            ]
        )
    )
    .to_numpy()
)

# Total interactions available for each customer
customer_category_affinity["customer_interaction_count"] = (
    customer_category_affinity
    .groupby("visitorid")["interaction_count"]
    .transform("sum")
)

# Simple evidence/confidence indicator
# More interactions = more confidence.
customer_category_affinity["evidence_ratio"] = (
    customer_category_affinity["interaction_count"]
    / customer_category_affinity["customer_interaction_count"]
)

# Number of categories observed for each customer
customer_category_affinity["num_categories"] = (
    customer_category_affinity
    .groupby("visitorid")["categoryid"]
    .transform("nunique")
)

display(
    customer_category_affinity.head(20)
)

,visitorid,categoryid,interaction_weight,customer_total_score,affinity_score,interaction_count,customer_interaction_count,evidence_ratio,num_categories
0,0,256,1.0,3.0,0.333333,1,3,0.333333,3
1,0,333,1.0,3.0,0.333333,1,3,0.333333,3
2,0,1188,1.0,3.0,0.333333,1,3,0.333333,3
3,1,1192,1.0,1.0,1.000000,1,1,1.000000,1
4,2,299,6.0,8.0,0.750000,6,8,0.750000,2
5,2,444,2.0,8.0,0.250000,2,8,0.250000,2
6,3,1171,1.0,1.0,1.000000,1,1,1.000000,1
7,5,646,1.0,1.0,1.000000,1,1,1.000000,1
8,6,342,8.0,8.0,1.000000,6,6,1.000000,1
9,7,642,1.0,2.0,0.500000,1,2,0.500000,2


In [72]:
# ============================================================
# HOW MANY CATEGORIES DOES EACH CUSTOMER INTERACT WITH?
# ============================================================

customer_category_count = (
    customer_category_affinity
    .groupby("visitorid")["categoryid"]
    .nunique()
)

print("Customers:", customer_category_count.shape[0])

display(
    customer_category_count
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nSummary:")
display(customer_category_count.describe())

Customers: 1076473


categoryid
1     984059
2      68125
3      13801
4       4747
5       2101
6       1092
7        656
8        435
9        284
10       219
11       161
12        99
13        85
14        59
15        50
16        41
17        34
18        36
19        25
20        25
Name: count, dtype: int64


Summary:


count    1.076473e+06
mean     1.156903e+00
std      1.997603e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.250000e+02
Name: categoryid, dtype: float64

In [73]:
# ============================================================
# RECENT / TIME-DECAYED CUSTOMER-CATEGORY AFFINITY
# ============================================================

import numpy as np
import pandas as pd

# Work only with events that have a known category
recent_affinity_events = enriched_events.dropna(
    subset=["categoryid"]
).copy()

# Make category ID integer
recent_affinity_events["categoryid"] = (
    recent_affinity_events["categoryid"].astype(int)
)

# Event strength
event_weights = {
    "view": 1.0,
    "addtocart": 3.0,
    "transaction": 5.0
}

recent_affinity_events["event_weight"] = (
    recent_affinity_events["event"]
    .map(event_weights)
    .fillna(0)
)

# Reference point = last observed event in the dataset
reference_date = enriched_events["datetime"].max()

# Age of each interaction in days
recent_affinity_events["age_days"] = (
    reference_date -
    recent_affinity_events["datetime"]
).dt.total_seconds() / (24 * 60 * 60)

# 30-day half-life
half_life_days = 30

# Time-decay factor
recent_affinity_events["time_decay"] = np.exp(
    -np.log(2) *
    recent_affinity_events["age_days"] /
    half_life_days
)

# Final time-aware interaction weight
recent_affinity_events["decayed_weight"] = (
    recent_affinity_events["event_weight"] *
    recent_affinity_events["time_decay"]
)

# Aggregate by customer + category
recent_category_scores = (
    recent_affinity_events
    .groupby(["visitorid", "categoryid"])["decayed_weight"]
    .sum()
    .reset_index(name="recent_score")
)

# Total recent score per customer
recent_category_scores["customer_recent_total"] = (
    recent_category_scores
    .groupby("visitorid")["recent_score"]
    .transform("sum")
)

# Normalize within customer
recent_category_scores["recent_affinity"] = (
    recent_category_scores["recent_score"] /
    recent_category_scores["customer_recent_total"]
)

# Sort by customer and strongest recent interest
recent_category_scores = (
    recent_category_scores
    .sort_values(
        ["visitorid", "recent_affinity"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("Customers with recent affinity:",
      recent_category_scores["visitorid"].nunique())

print("Customer-category rows:",
      len(recent_category_scores))

print("\nSample:")
display(recent_category_scores.head(20))

Customers with recent affinity: 1076473
Customer-category rows: 1245375

Sample:


,visitorid,categoryid,recent_score,customer_recent_total,recent_affinity
0,0,333,0.865474,2.596309,0.333348
1,0,256,0.865437,2.596309,0.333334
2,0,1188,0.865398,2.596309,0.333319
3,1,1192,0.441510,0.441510,1.000000
4,2,299,2.306788,3.075832,0.749972
5,2,444,0.769043,3.075832,0.250028
6,3,1171,0.331207,0.331207,1.000000
7,5,646,0.232982,0.232982,1.000000
8,6,342,5.187944,5.187944,1.000000
9,7,642,0.055753,0.109059,0.511221


In [74]:
# ============================================================
# CUSTOMER RECENCY + FREQUENCY FEATURES
# ============================================================

# Reference date = last timestamp available in the dataset
reference_date = enriched_events["datetime"].max()

# ------------------------------------------------------------
# Basic customer activity features
# ------------------------------------------------------------

customer_recency_frequency = (
    enriched_events
    .groupby("visitorid")
    .agg(
        total_interactions=("event", "size"),
        total_views=("event", lambda x: (x == "view").sum()),
        total_cart_adds=("event", lambda x: (x == "addtocart").sum()),
        total_transactions=("event", lambda x: (x == "transaction").sum()),
        unique_products=("itemid", "nunique"),
        first_activity=("datetime", "min"),
        last_activity=("datetime", "max")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Last purchase date
# ------------------------------------------------------------

last_purchase = (
    enriched_events[
        enriched_events["event"] == "transaction"
    ]
    .groupby("visitorid")["datetime"]
    .max()
    .rename("last_purchase")
    .reset_index()
)

customer_recency_frequency = customer_recency_frequency.merge(
    last_purchase,
    on="visitorid",
    how="left"
)

# ------------------------------------------------------------
# Recency
# ------------------------------------------------------------

customer_recency_frequency["recency_days"] = (
    reference_date -
    customer_recency_frequency["last_activity"]
).dt.total_seconds() / (24 * 60 * 60)

customer_recency_frequency["purchase_recency_days"] = (
    reference_date -
    customer_recency_frequency["last_purchase"]
).dt.total_seconds() / (24 * 60 * 60)

# Customers who never purchased get missing purchase recency.
customer_recency_frequency["purchase_recency_days"] = (
    customer_recency_frequency["purchase_recency_days"]
    .fillna(-1)
)

# ------------------------------------------------------------
# Active duration
# ------------------------------------------------------------

customer_recency_frequency["active_days"] = (
    customer_recency_frequency["last_activity"] -
    customer_recency_frequency["first_activity"]
).dt.total_seconds() / (24 * 60 * 60)

# ------------------------------------------------------------
# Frequency-like measures
# ------------------------------------------------------------

customer_recency_frequency["interactions_per_active_day"] = (
    customer_recency_frequency["total_interactions"] /
    customer_recency_frequency["active_days"].clip(lower=1)
)

customer_recency_frequency["transactions_per_active_day"] = (
    customer_recency_frequency["total_transactions"] /
    customer_recency_frequency["active_days"].clip(lower=1)
)

# ------------------------------------------------------------
# Engagement ratios
# ------------------------------------------------------------

customer_recency_frequency["cart_rate"] = (
    customer_recency_frequency["total_cart_adds"] /
    customer_recency_frequency["total_views"].replace(0, np.nan)
).fillna(0)

customer_recency_frequency["purchase_rate"] = (
    customer_recency_frequency["total_transactions"] /
    customer_recency_frequency["total_views"].replace(0, np.nan)
).fillna(0)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("Customers:", len(customer_recency_frequency))

display(
    customer_recency_frequency.head(20)
)

print("\nNumeric summary:")
display(
    customer_recency_frequency[
        [
            "total_interactions",
            "total_views",
            "total_cart_adds",
            "total_transactions",
            "unique_products",
            "recency_days",
            "purchase_recency_days",
            "active_days",
            "interactions_per_active_day",
            "transactions_per_active_day",
            "cart_rate",
            "purchase_rate"
        ]
    ].describe()
)

Customers: 1407580


,visitorid,total_interactions,total_views,total_cart_adds,total_transactions,unique_products,first_activity,last_activity,last_purchase,recency_days,purchase_recency_days,active_days,interactions_per_active_day,transactions_per_active_day,cart_rate,purchase_rate
0,0,3,3,0,0,3,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,NaT,6.253132,-1.0,0.003793,3.000000,0.0,0.0,0.0
1,1,1,1,0,0,1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,NaT,35.384506,-1.0,0.000000,1.000000,0.0,0.0,0.0
2,2,8,8,0,0,4,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,NaT,41.360300,-1.0,0.020293,8.000000,0.0,0.0,0.0
3,3,1,1,0,0,1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,NaT,47.825839,-1.0,0.000000,1.000000,0.0,0.0,0.0
4,4,1,1,0,0,1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,NaT,2.232878,-1.0,0.000000,1.000000,0.0,0.0,0.0
5,5,1,1,0,0,1,2015-07-17 01:45:56.439,2015-07-17 01:45:56.439,NaT,63.051289,-1.0,0.000000,1.000000,0.0,0.0,0.0
6,6,6,5,1,0,3,2015-08-30 06:03:48.202,2015-08-31 03:21:25.697,NaT,17.984978,-1.0,0.887240,6.000000,0.0,0.2,0.0
7,7,3,3,0,0,3,2015-05-14 05:39:36.753,2015-05-16 04:20:39.214,NaT,124.943849,-1.0,1.945167,1.542284,0.0,0.0,0.0
8,8,1,1,0,0,1,2015-05-31 00:01:53.812,2015-05-31 00:01:53.812,NaT,110.123541,-1.0,0.000000,1.000000,0.0,0.0,0.0
9,9,1,1,0,0,1,2015-07-08 17:36:47.285,2015-07-08 17:36:47.285,NaT,71.390978,-1.0,0.000000,1.000000,0.0,0.0,0.0



Numeric summary:


,total_interactions,total_views,total_cart_adds,total_transactions,unique_products,recency_days,purchase_recency_days,active_days,interactions_per_active_day,transactions_per_active_day,cart_rate,purchase_rate
count,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06
mean,1.958042e+00,1.892832e+00,4.925617e-02,1.595433e-02,1.524019e+00,6.832110e+01,-4.066750e-01,2.375490e+00,1.445364e+00,7.904144e-03,1.359857e-02,3.498930e-03
std,1.258049e+01,1.099370e+01,1.165057e+00,8.260909e-01,7.143724e+00,3.916082e+01,7.392680e+00,1.163081e+01,1.959294e+00,1.334588e-01,1.294045e-01,5.283566e-02
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,1.465966e-02,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,3.542847e+01,-1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,6.707738e+01,-1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.020853e+02,-1.000000e+00,5.442274e-04,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,7.757000e+03,6.479000e+03,7.190000e+02,5.590000e+02,3.814000e+03,1.379997e+02,1.379809e+02,1.379729e+02,3.100000e+02,3.621850e+01,3.100000e+01,6.000000e+00


In [75]:
# ============================================================
# CUSTOMER INTELLIGENCE MASTER FEATURE TABLE
# ============================================================

# 1. Copy the recency/frequency table
customer_profile_features = customer_recency_frequency.copy()

# ------------------------------------------------------------
# 2. Historical affinity evidence
# ------------------------------------------------------------

historical_affinity_summary = (
    customer_category_affinity
    .groupby("visitorid")
    .agg(
        num_categories=("categoryid", "nunique"),
        total_category_interactions=("interaction_count", "sum"),
        max_affinity=("affinity_score", "max"),
        avg_affinity=("affinity_score", "mean"),
        max_affinity_interactions=("interaction_count", "max")
    )
    .reset_index()
)

customer_profile_features = customer_profile_features.merge(
    historical_affinity_summary,
    on="visitorid",
    how="left"
)

# ------------------------------------------------------------
# 3. Recent affinity evidence
# ------------------------------------------------------------

recent_affinity_summary = (
    recent_category_scores
    .groupby("visitorid")
    .agg(
        recent_num_categories=("categoryid", "nunique"),
        max_recent_affinity=("recent_affinity", "max"),
        max_recent_score=("recent_score", "max")
    )
    .reset_index()
)

customer_profile_features = customer_profile_features.merge(
    recent_affinity_summary,
    on="visitorid",
    how="left"
)

# ------------------------------------------------------------
# 4. Purchase indicator
# ------------------------------------------------------------

customer_profile_features["has_purchased"] = (
    customer_profile_features["total_transactions"] > 0
).astype(int)

# ------------------------------------------------------------
# 5. Evidence level
# ------------------------------------------------------------

def assign_evidence_level(row):

    interactions = row["total_interactions"]

    if interactions <= 1:
        return "Low"

    elif interactions <= 5:
        return "Limited"

    elif interactions <= 20:
        return "Moderate"

    else:
        return "Strong"


customer_profile_features["profile_evidence"] = (
    customer_profile_features.apply(
        assign_evidence_level,
        axis=1
    )
)

# ------------------------------------------------------------
# 6. Clean missing affinity values
# ------------------------------------------------------------

affinity_columns = [
    "num_categories",
    "total_category_interactions",
    "max_affinity",
    "avg_affinity",
    "max_affinity_interactions",
    "recent_num_categories",
    "max_recent_affinity",
    "max_recent_score"
]

customer_profile_features[affinity_columns] = (
    customer_profile_features[affinity_columns]
    .fillna(0)
)

# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("Customer profile rows:",
      len(customer_profile_features))

print("\nEvidence-level distribution:")
display(
    customer_profile_features["profile_evidence"]
    .value_counts()
)

print("\nSample customer profiles:")
display(
    customer_profile_features[
        [
            "visitorid",
            "total_interactions",
            "total_views",
            "total_cart_adds",
            "total_transactions",
            "unique_products",
            "recency_days",
            "has_purchased",
            "num_categories",
            "max_affinity",
            "max_recent_affinity",
            "profile_evidence"
        ]
    ].head(20)
)

Customer profile rows: 1407580

Evidence-level distribution:


profile_evidence
Low         1001560
Limited      347367
Moderate      52596
Strong         6057
Name: count, dtype: int64


Sample customer profiles:


,visitorid,total_interactions,total_views,total_cart_adds,total_transactions,unique_products,recency_days,has_purchased,num_categories,max_affinity,max_recent_affinity,profile_evidence
0,0,3,3,0,0,3,6.253132,0,3.0,0.333333,0.333348,Limited
1,1,1,1,0,0,1,35.384506,0,1.0,1.000000,1.000000,Low
2,2,8,8,0,0,4,41.360300,0,2.0,0.750000,0.749972,Moderate
3,3,1,1,0,0,1,47.825839,0,1.0,1.000000,1.000000,Low
4,4,1,1,0,0,1,2.232878,0,0.0,0.000000,0.000000,Low
5,5,1,1,0,0,1,63.051289,0,1.0,1.000000,1.000000,Low
6,6,6,5,1,0,3,17.984978,0,1.0,1.000000,1.000000,Moderate
7,7,3,3,0,0,3,124.943849,0,2.0,0.500000,0.511221,Limited
8,8,1,1,0,0,1,110.123541,0,0.0,0.000000,0.000000,Low
9,9,1,1,0,0,1,71.390978,0,1.0,1.000000,1.000000,Low


In [76]:
# ============================================================
# PROFILE CONFIDENCE SCORE
# ============================================================

profile_df = customer_profile_features.copy()

# Behaviour evidence
interaction_score = np.log1p(
    profile_df["total_interactions"]
)

purchase_score = np.log1p(
    profile_df["total_transactions"]
)

category_score = np.log1p(
    profile_df["num_categories"]
)

product_score = np.log1p(
    profile_df["unique_products"]
)

# Combine evidence signals
raw_confidence = (
    0.40 * interaction_score +
    0.30 * purchase_score +
    0.20 * category_score +
    0.10 * product_score
)

# Normalize to 0–1
min_score = raw_confidence.min()
max_score = raw_confidence.max()

profile_df["profile_confidence"] = (
    (raw_confidence - min_score) /
    (max_score - min_score)
    if max_score > min_score
    else 0
)

# Confidence bands
def confidence_band(score):
    if score < 0.20:
        return "Low"
    elif score < 0.50:
        return "Medium"
    elif score < 0.75:
        return "High"
    else:
        return "Very High"

profile_df["confidence_band"] = (
    profile_df["profile_confidence"]
    .apply(confidence_band)
)

print("Confidence distribution:")
display(
    profile_df["confidence_band"]
    .value_counts()
)

print("\nSample:")
display(
    profile_df[
        [
            "visitorid",
            "total_interactions",
            "total_transactions",
            "unique_products",
            "num_categories",
            "profile_evidence",
            "profile_confidence",
            "confidence_band"
        ]
    ].head(20)
)

Confidence distribution:


confidence_band
Low          1401354
Medium          6086
High             117
Very High         23
Name: count, dtype: int64


Sample:


,visitorid,total_interactions,total_transactions,unique_products,num_categories,profile_evidence,profile_confidence,confidence_band
0,0,3,0,3,3.0,Limited,0.086498,Low
1,1,1,0,1,1.0,Low,0.019222,Low
2,2,8,0,4,2.0,Moderate,0.126590,Low
3,3,1,0,1,1.0,Low,0.019222,Low
4,4,1,0,1,0.0,Low,0.000000,Low
5,5,1,0,1,1.0,Low,0.019222,Low
6,6,6,0,3,1.0,Moderate,0.098313,Low
7,7,3,0,3,2.0,Limited,0.078520,Low
8,8,1,0,1,0.0,Low,0.000000,Low
9,9,1,0,1,1.0,Low,0.019222,Low


In [77]:
# ============================================================
# CUSTOMER EVIDENCE TIER
# ============================================================

profile_df = customer_profile_features.copy()

# ------------------------------------------------------------
# Categorized interaction count
# ------------------------------------------------------------

categorized_interactions = (
    enriched_events
    .dropna(subset=["categoryid"])
    .groupby("visitorid")
    .size()
    .rename("categorized_interactions")
    .reset_index()
)

profile_df = profile_df.merge(
    categorized_interactions,
    on="visitorid",
    how="left"
)

profile_df["categorized_interactions"] = (
    profile_df["categorized_interactions"]
    .fillna(0)
    .astype(int)
)

# ------------------------------------------------------------
# Category coverage
# ------------------------------------------------------------

profile_df["category_coverage"] = (
    profile_df["categorized_interactions"] /
    profile_df["total_interactions"]
).fillna(0)

# ------------------------------------------------------------
# Evidence tier
# ------------------------------------------------------------

def evidence_tier(row):

    interactions = row["total_interactions"]
    purchases = row["total_transactions"]
    carts = row["total_cart_adds"]

    # Tier 1: extremely little evidence
    if interactions <= 2 and purchases == 0:
        return "Cold / New"

    # Tier 2: some behavioural evidence
    elif interactions <= 10 and purchases <= 1:
        return "Developing"

    # Tier 3: stronger behavioural evidence
    elif interactions <= 50 or purchases <= 3:
        return "Established"

    # Tier 4: highly engaged customer
    else:
        return "Highly Engaged"


profile_df["evidence_tier"] = (
    profile_df.apply(evidence_tier, axis=1)
)

# ------------------------------------------------------------
# Display distribution
# ------------------------------------------------------------

print("Evidence tier distribution:")
display(
    profile_df["evidence_tier"]
    .value_counts()
)

print("\nSample profiles:")
display(
    profile_df[
        [
            "visitorid",
            "total_interactions",
            "categorized_interactions",
            "category_coverage",
            "total_views",
            "total_cart_adds",
            "total_transactions",
            "unique_products",
            "num_categories",
            "recency_days",
            "has_purchased",
            "evidence_tier"
        ]
    ].head(20)
)

Evidence tier distribution:


evidence_tier
Cold / New        1206897
Developing         180278
Established         20161
Highly Engaged        244
Name: count, dtype: int64


Sample profiles:


,visitorid,total_interactions,categorized_interactions,category_coverage,total_views,total_cart_adds,total_transactions,unique_products,num_categories,recency_days,has_purchased,evidence_tier
0,0,3,3,1.000000,3,0,0,3,3.0,6.253132,0,Developing
1,1,1,1,1.000000,1,0,0,1,1.0,35.384506,0,Cold / New
2,2,8,8,1.000000,8,0,0,4,2.0,41.360300,0,Developing
3,3,1,1,1.000000,1,0,0,1,1.0,47.825839,0,Cold / New
4,4,1,0,0.000000,1,0,0,1,0.0,2.232878,0,Cold / New
5,5,1,1,1.000000,1,0,0,1,1.0,63.051289,0,Cold / New
6,6,6,6,1.000000,5,1,0,3,1.0,17.984978,0,Developing
7,7,3,2,0.666667,3,0,0,3,2.0,124.943849,0,Developing
8,8,1,0,0.000000,1,0,0,1,0.0,110.123541,0,Cold / New
9,9,1,1,1.000000,1,0,0,1,1.0,71.390978,0,Cold / New


In [78]:
# ============================================================
# BEHAVIOURAL DISTRIBUTIONS FOR PERSONA DESIGN
# ============================================================

persona_stats = profile_df.copy()

# ------------------------------------------------------------
# Transaction distribution
# ------------------------------------------------------------

print("Transaction count distribution:")
display(
    persona_stats["total_transactions"]
    .value_counts()
    .sort_index()
    .head(20)
)

# ------------------------------------------------------------
# Cart distribution
# ------------------------------------------------------------

print("\nCart-add distribution:")
display(
    persona_stats["total_cart_adds"]
    .value_counts()
    .sort_index()
    .head(20)
)

# ------------------------------------------------------------
# View distribution
# ------------------------------------------------------------

print("\nView-count percentiles:")
display(
    persona_stats["total_views"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.999]
    )
)

# ------------------------------------------------------------
# Transaction percentiles among purchasers only
# ------------------------------------------------------------

purchasers = persona_stats[
    persona_stats["total_transactions"] > 0
]

print("\nTransaction percentiles among purchasers:")
display(
    purchasers["total_transactions"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

# ------------------------------------------------------------
# Category diversity percentiles
# ------------------------------------------------------------

active_category_customers = persona_stats[
    persona_stats["num_categories"] > 0
]

print("\nCategory-count percentiles:")
display(
    active_category_customers["num_categories"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Transaction count distribution:


total_transactions
0     1395861
1        9143
2        1549
3         435
4         215
5          83
6          67
7          26
8          27
9          16
10         20
11          9
12          5
13         13
14         14
15          4
16          6
17          4
18          4
19          3
Name: count, dtype: int64


Cart-add distribution:


total_cart_adds
0     1369858
1       28647
2        4946
3        1630
4         845
5         440
6         295
7         171
8         144
9          95
10         72
11         61
12         31
13         43
14         30
15         22
16         22
17         13
18         17
19         10
Name: count, dtype: int64


View-count percentiles:


0.500     1.0
0.750     2.0
0.900     3.0
0.950     5.0
0.990    12.0
0.999    43.0
Name: total_views, dtype: float64


Transaction percentiles among purchasers:


0.50     1.0
0.75     1.0
0.90     2.0
0.95     4.0
0.99    13.0
Name: total_transactions, dtype: float64


Category-count percentiles:


0.50    1.0
0.75    1.0
0.90    1.0
0.95    2.0
0.99    3.0
Name: num_categories, dtype: float64

In [79]:
# ============================================================
# PERSONA CANDIDATE DISTRIBUTIONS
# ============================================================

df = profile_df.copy()

# Candidate behavioural flags
df["is_browser"] = (
    (df["total_views"] >= 3) &
    (df["total_transactions"] == 0) &
    (df["total_cart_adds"] <= 1)
)

df["is_engaged"] = (
    (
        (df["total_cart_adds"] >= 1) |
        (df["total_views"] >= 5)
    ) &
    (df["total_transactions"] <= 1)
)

df["is_one_time_purchaser"] = (
    df["total_transactions"] == 1
)

df["is_repeat_purchaser"] = (
    df["total_transactions"].between(2, 4)
)

df["is_loyal_purchaser"] = (
    df["total_transactions"] >= 5
)

df["is_multi_category"] = (
    df["num_categories"] >= 3
)

print("Candidate persona/behaviour counts:")
print("Browser / Explorer:", df["is_browser"].sum())
print("Engaged Shopper:", df["is_engaged"].sum())
print("One-Time Purchaser:", df["is_one_time_purchaser"].sum())
print("Repeat Purchaser:", df["is_repeat_purchaser"].sum())
print("Loyal Purchaser:", df["is_loyal_purchaser"].sum())
print("Multi-Category:", df["is_multi_category"].sum())

print("\nOverlaps:")
display(
    df[
        [
            "is_browser",
            "is_engaged",
            "is_one_time_purchaser",
            "is_repeat_purchaser",
            "is_loyal_purchaser",
            "is_multi_category"
        ]
    ].sum()
)

Candidate persona/behaviour counts:
Browser / Explorer: 179533
Engaged Shopper: 98589
One-Time Purchaser: 9143
Repeat Purchaser: 2199
Loyal Purchaser: 377
Multi-Category: 24289

Overlaps:


is_browser               179533
is_engaged                98589
is_one_time_purchaser      9143
is_repeat_purchaser        2199
is_loyal_purchaser          377
is_multi_category         24289
dtype: int64

In [80]:
# ============================================================
# PERSONA FLAG OVERLAP ANALYSIS
# ============================================================

flag_columns = [
    "is_browser",
    "is_engaged",
    "is_one_time_purchaser",
    "is_repeat_purchaser",
    "is_loyal_purchaser",
    "is_multi_category"
]

# Number of active persona/behaviour flags per customer
df["num_active_flags"] = df[flag_columns].sum(axis=1)

print("How many customers satisfy multiple conditions?")
display(
    df["num_active_flags"]
    .value_counts()
    .sort_index()
)

# Pairwise overlap matrix
print("\nPairwise overlap matrix:")
overlap_matrix = pd.DataFrame(
    index=flag_columns,
    columns=flag_columns,
    dtype=int
)

for col1 in flag_columns:
    for col2 in flag_columns:
        overlap_matrix.loc[col1, col2] = (
            df[col1] & df[col2]
        ).sum()

display(overlap_matrix)

# Example customers satisfying multiple conditions
print("\nSample customers with 2+ active flags:")
display(
    df.loc[
        df["num_active_flags"] >= 2,
        [
            "visitorid",
            "total_views",
            "total_cart_adds",
            "total_transactions",
            "unique_products",
            "num_categories",
            "recency_days",
            "is_browser",
            "is_engaged",
            "is_one_time_purchaser",
            "is_repeat_purchaser",
            "is_loyal_purchaser",
            "is_multi_category"
        ]
    ].head(20)
)

How many customers satisfy multiple conditions?


num_active_flags
0    1198088
1     120570
2      73206
3      15716
Name: count, dtype: int64


Pairwise overlap matrix:


,is_browser,is_engaged,is_one_time_purchaser,is_repeat_purchaser,is_loyal_purchaser,is_multi_category
is_browser,179533.0,71978.0,0.0,0.0,0.0,19979.0
is_engaged,71978.0,98589.0,8371.0,0.0,0.0,17629.0
is_one_time_purchaser,0.0,8371.0,9143.0,0.0,0.0,1047.0
is_repeat_purchaser,0.0,0.0,0.0,2199.0,0.0,1009.0
is_loyal_purchaser,0.0,0.0,0.0,0.0,377.0,341.0
is_multi_category,19979.0,17629.0,1047.0,1009.0,341.0,24289.0



Sample customers with 2+ active flags:


,visitorid,total_views,total_cart_adds,total_transactions,unique_products,num_categories,recency_days,is_browser,is_engaged,is_one_time_purchaser,is_repeat_purchaser,is_loyal_purchaser,is_multi_category
0,0,3,0,0,3,3.0,6.253132,True,False,False,False,False,True
2,2,8,0,0,4,2.0,41.360300,True,True,False,False,False,False
6,6,5,1,0,3,1.0,17.984978,True,True,False,False,False,False
37,37,8,0,0,2,2.0,35.461949,True,True,False,False,False,False
51,51,6,0,0,5,2.0,12.147034,True,True,False,False,False,False
54,54,11,0,0,6,2.0,0.080792,True,True,False,False,False,False
64,64,7,0,0,6,3.0,86.078234,True,True,False,False,False,True
74,74,7,0,0,5,2.0,37.844652,True,True,False,False,False,False
75,75,26,0,0,2,1.0,58.478235,True,True,False,False,False,False
97,97,8,0,0,3,1.0,60.730301,True,True,False,False,False,False


In [81]:
# ============================================================
# FINAL PRIMARY PERSONA + BEHAVIOURAL ATTRIBUTES
# ============================================================

persona_df = profile_df.copy()

# ------------------------------------------------------------
# PRIMARY PERSONA
# Priority:
# Loyal -> Repeat -> One-Time Purchaser
# -> Engaged -> Browser -> New/Unknown
# ------------------------------------------------------------

def assign_primary_persona(row):

    transactions = row["total_transactions"]
    views = row["total_views"]
    carts = row["total_cart_adds"]
    interactions = row["total_interactions"]

    # 1. Loyal purchaser
    if transactions >= 5:
        return "Loyal Purchaser"

    # 2. Repeat purchaser
    elif transactions >= 2:
        return "Repeat Purchaser"

    # 3. One-time purchaser
    elif transactions == 1:
        return "One-Time Purchaser"

    # 4. Engaged shopper
    elif interactions >= 5 or carts >= 1:
        return "Engaged Shopper"

    # 5. Browser / Explorer
    elif views >= 3:
        return "Browser / Explorer"

    # 6. New / Unknown
    else:
        return "New / Unknown"


persona_df["primary_persona"] = (
    persona_df.apply(
        assign_primary_persona,
        axis=1
    )
)

# ------------------------------------------------------------
# BEHAVIOURAL ATTRIBUTES
# These are NOT mutually exclusive.
# ------------------------------------------------------------

persona_df["is_multi_category"] = (
    persona_df["num_categories"] >= 3
)

persona_df["is_recently_active"] = (
    persona_df["recency_days"] <= 7
)

persona_df["is_highly_active"] = (
    persona_df["total_interactions"] >= 20
)

persona_df["is_cart_heavy"] = (
    (
        persona_df["total_cart_adds"] >= 2
    ) &
    (
        persona_df["total_transactions"] == 0
    )
)

persona_df["is_repeat_purchaser"] = (
    persona_df["total_transactions"] >= 2
)

# ------------------------------------------------------------
# PERSONA DISTRIBUTION
# ------------------------------------------------------------

print("Primary persona distribution:")
display(
    persona_df["primary_persona"]
    .value_counts()
)

# ------------------------------------------------------------
# ATTRIBUTE DISTRIBUTION
# ------------------------------------------------------------

print("\nBehavioural attributes:")
print(
    "Multi-category:",
    persona_df["is_multi_category"].sum()
)

print(
    "Recently active:",
    persona_df["is_recently_active"].sum()
)

print(
    "Highly active:",
    persona_df["is_highly_active"].sum()
)

print(
    "Cart-heavy:",
    persona_df["is_cart_heavy"].sum()
)

print(
    "Repeat purchaser:",
    persona_df["is_repeat_purchaser"].sum()
)

# ------------------------------------------------------------
# SAMPLE
# ------------------------------------------------------------

print("\nSample customer personas:")
display(
    persona_df[
        [
            "visitorid",
            "primary_persona",
            "profile_evidence",
            "total_interactions",
            "total_views",
            "total_cart_adds",
            "total_transactions",
            "unique_products",
            "num_categories",
            "recency_days",
            "is_multi_category",
            "is_recently_active",
            "is_highly_active"
        ]
    ].head(30)
)

Primary persona distribution:


primary_persona
New / Unknown         1198088
Browser / Explorer     107555
Engaged Shopper         90218
One-Time Purchaser       9143
Repeat Purchaser         2199
Loyal Purchaser           377
Name: count, dtype: int64


Behavioural attributes:
Multi-category: 24289
Recently active: 67930
Highly active: 6610
Cart-heavy: 5823
Repeat purchaser: 2576

Sample customer personas:


,visitorid,primary_persona,profile_evidence,total_interactions,total_views,total_cart_adds,total_transactions,unique_products,num_categories,recency_days,is_multi_category,is_recently_active,is_highly_active
0,0,Browser / Explorer,Limited,3,3,0,0,3,3.0,6.253132,True,True,False
1,1,New / Unknown,Low,1,1,0,0,1,1.0,35.384506,False,False,False
2,2,Engaged Shopper,Moderate,8,8,0,0,4,2.0,41.360300,False,False,False
3,3,New / Unknown,Low,1,1,0,0,1,1.0,47.825839,False,False,False
4,4,New / Unknown,Low,1,1,0,0,1,0.0,2.232878,False,True,False
5,5,New / Unknown,Low,1,1,0,0,1,1.0,63.051289,False,False,False
6,6,Engaged Shopper,Moderate,6,5,1,0,3,1.0,17.984978,False,False,False
7,7,Browser / Explorer,Limited,3,3,0,0,3,2.0,124.943849,False,False,False
8,8,New / Unknown,Low,1,1,0,0,1,0.0,110.123541,False,False,False
9,9,New / Unknown,Low,1,1,0,0,1,1.0,71.390978,False,False,False


In [82]:
# ============================================================
# FINAL CUSTOMER DIGITAL TWIN / PERSONA PROFILE
# ============================================================

# Start from the persona table
digital_twin = persona_df.copy()

# ------------------------------------------------------------
# 1. TOP HISTORICAL CATEGORIES
# ------------------------------------------------------------

historical_top = (
    customer_category_affinity
    .sort_values(
        ["visitorid", "affinity_score"],
        ascending=[True, False]
    )
    .copy()
)

historical_top["category_rank"] = (
    historical_top
    .groupby("visitorid")
    .cumcount() + 1
)

historical_top = historical_top[
    historical_top["category_rank"] <= 3
]

historical_pivot = (
    historical_top
    .pivot(
        index="visitorid",
        columns="category_rank",
        values=["categoryid", "affinity_score"]
    )
)

historical_pivot.columns = [
    f"{'top_category' if col[0] == 'categoryid' else 'top_category_affinity'}_{col[1]}"
    for col in historical_pivot.columns
]

historical_pivot = historical_pivot.reset_index()

digital_twin = digital_twin.merge(
    historical_pivot,
    on="visitorid",
    how="left"
)

# ------------------------------------------------------------
# 2. TOP RECENT CATEGORIES
# ------------------------------------------------------------

recent_top = (
    recent_category_scores
    .sort_values(
        ["visitorid", "recent_affinity"],
        ascending=[True, False]
    )
    .copy()
)

recent_top["category_rank"] = (
    recent_top
    .groupby("visitorid")
    .cumcount() + 1
)

recent_top = recent_top[
    recent_top["category_rank"] <= 2
]

recent_pivot = (
    recent_top
    .pivot(
        index="visitorid",
        columns="category_rank",
        values=["categoryid", "recent_affinity"]
    )
)

recent_pivot.columns = [
    f"{'recent_category' if col[0] == 'categoryid' else 'recent_category_affinity'}_{col[1]}"
    for col in recent_pivot.columns
]

recent_pivot = recent_pivot.reset_index()

digital_twin = digital_twin.merge(
    recent_pivot,
    on="visitorid",
    how="left"
)

# ------------------------------------------------------------
# 3. CLEAN CATEGORY COLUMNS
# ------------------------------------------------------------

category_columns = [
    "top_category_1",
    "top_category_2",
    "top_category_3",
    "top_category_affinity_1",
    "top_category_affinity_2",
    "top_category_affinity_3",
    "recent_category_1",
    "recent_category_2",
    "recent_category_affinity_1",
    "recent_category_affinity_2"
]

for col in category_columns:
    if col in digital_twin.columns:
        if "category_" in col and "affinity" not in col:
            digital_twin[col] = digital_twin[col].astype("Int64")
        else:
            digital_twin[col] = digital_twin[col].fillna(0)

# ------------------------------------------------------------
# 4. SELECT FINAL CUSTOMER PROFILE FIELDS
# ------------------------------------------------------------

profile_columns = [
    "visitorid",
    "primary_persona",
    "profile_evidence",
    "evidence_tier",
    "total_interactions",
    "total_views",
    "total_cart_adds",
    "total_transactions",
    "unique_products",
    "recency_days",
    "purchase_recency_days",
    "num_categories",
    "max_affinity",
    "max_recent_affinity",
    "has_purchased",
    "is_multi_category",
    "is_recently_active",
    "is_highly_active",
    "is_cart_heavy",
    "top_category_1",
    "top_category_affinity_1",
    "top_category_2",
    "top_category_affinity_2",
    "top_category_3",
    "top_category_affinity_3",
    "recent_category_1",
    "recent_category_affinity_1",
    "recent_category_2",
    "recent_category_affinity_2"
]

profile_columns = [
    col for col in profile_columns
    if col in digital_twin.columns
]

digital_twin = digital_twin[profile_columns]

print("Digital Twin rows:", len(digital_twin))
print("Digital Twin columns:", len(digital_twin.columns))

display(digital_twin.head(20))

Digital Twin rows: 1407580
Digital Twin columns: 29


,visitorid,primary_persona,profile_evidence,evidence_tier,total_interactions,total_views,total_cart_adds,total_transactions,unique_products,recency_days,...,top_category_1,top_category_affinity_1,top_category_2,top_category_affinity_2,top_category_3,top_category_affinity_3,recent_category_1,recent_category_affinity_1,recent_category_2,recent_category_affinity_2
0,0,Browser / Explorer,Limited,Developing,3,3,0,0,3,6.253132,...,256,0.333333,333,0.333333,1188,0.333333,333,0.333348,256,0.333334
1,1,New / Unknown,Low,Cold / New,1,1,0,0,1,35.384506,...,1192,1.000000,<NA>,0.000000,<NA>,0.000000,1192,1.000000,<NA>,0.000000
2,2,Engaged Shopper,Moderate,Developing,8,8,0,0,4,41.360300,...,299,0.750000,444,0.250000,<NA>,0.000000,299,0.749972,444,0.250028
3,3,New / Unknown,Low,Cold / New,1,1,0,0,1,47.825839,...,1171,1.000000,<NA>,0.000000,<NA>,0.000000,1171,1.000000,<NA>,0.000000
4,4,New / Unknown,Low,Cold / New,1,1,0,0,1,2.232878,...,<NA>,0.000000,<NA>,0.000000,<NA>,0.000000,<NA>,0.000000,<NA>,0.000000
5,5,New / Unknown,Low,Cold / New,1,1,0,0,1,63.051289,...,646,1.000000,<NA>,0.000000,<NA>,0.000000,646,1.000000,<NA>,0.000000
6,6,Engaged Shopper,Moderate,Developing,6,5,1,0,3,17.984978,...,342,1.000000,<NA>,0.000000,<NA>,0.000000,342,1.000000,<NA>,0.000000
7,7,Browser / Explorer,Limited,Developing,3,3,0,0,3,124.943849,...,642,0.500000,1114,0.500000,<NA>,0.000000,642,0.511221,1114,0.488779
8,8,New / Unknown,Low,Cold / New,1,1,0,0,1,110.123541,...,<NA>,0.000000,<NA>,0.000000,<NA>,0.000000,<NA>,0.000000,<NA>,0.000000
9,9,New / Unknown,Low,Cold / New,1,1,0,0,1,71.390978,...,191,1.000000,<NA>,0.000000,<NA>,0.000000,191,1.000000,<NA>,0.000000
